In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [2]:
class HyperparameterTuner:

    def __init__(
        self,
        X_train_lr,
        y_train_lr,
        X_train_dt,
        y_train_dt,
        scale_pos_weight,
    ):
        # Linear data split
        self.X_train_lr = X_train_lr
        self.y_train_lr = y_train_lr

        # Tree data split
        self.X_train_dt = X_train_dt
        self.y_train_dt = y_train_dt

        self.scale_pos_weight = scale_pos_weight

        # Storage for best fitted estimators and CV scores
        self.best_estimators = {}
        self.cv_results_summary = []

    def _tune_logistic_regression(self):
        param_grid = {
            "C": [0.01, 0.1, 1.0, 10.0],
            "solver": ["liblinear", "lbfgs"],
        }
        grid = GridSearchCV(
            estimator=LogisticRegression(
                class_weight="balanced", random_state=42, max_iter=1000
            ),
            param_grid=param_grid,
            scoring="roc_auc",
            cv=5,
            n_jobs=-1,
        )
        grid.fit(self.X_train_lr, self.y_train_lr)

        self.best_estimators["Logistic Regression"] = grid.best_estimator_
        self.cv_results_summary.append(
            {
                "Model": "Logistic Regression",
                "Best ROC-AUC (CV)": round(grid.best_score_, 4),
                "Best Params": grid.best_params_,
            }
        )

    def _tune_random_forest(self):
        param_grid = {
            "n_estimators": [100, 200],
            "max_depth": [5, 10, None],
            "min_samples_split": [2, 5, 10],
        }
        grid = GridSearchCV(
            estimator=RandomForestClassifier(
                class_weight="balanced", random_state=42
            ),
            param_grid=param_grid,
            scoring="roc_auc",
            cv=5,
            n_jobs=-1,
        )
        grid.fit(self.X_train_dt, self.y_train_dt)

        self.best_estimators["Random Forest"] = grid.best_estimator_
        self.cv_results_summary.append(
            {
                "Model": "Random Forest",
                "Best ROC-AUC (CV)": round(grid.best_score_, 4),
                "Best Params": grid.best_params_,
            }
        )

    def _tune_xgboost(self):
        param_grid = {
            "n_estimators": [100, 200],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.1, 0.2],
        }
        grid = GridSearchCV(
            estimator=XGBClassifier(
                scale_pos_weight=self.scale_pos_weight,
                random_state=42,
                eval_metric="logloss",
            ),
            param_grid=param_grid,
            scoring="roc_auc",
            cv=5,
            n_jobs=-1,
        )
        grid.fit(self.X_train_dt, self.y_train_dt)

        self.best_estimators["XGBoost"] = grid.best_estimator_
        self.cv_results_summary.append(
            {
                "Model": "XGBoost",
                "Best ROC-AUC (CV)": round(grid.best_score_, 4),
                "Best Params": grid.best_params_,
            }
        )

    def run_tuning(self):
        print("1/3 Tuning hyperparameters (Logistic Regression)...")
        self._tune_logistic_regression()

        print("2/3 Tuning hyperparameters (Random Forest)...")
        self._tune_random_forest()

        print("3/3 Tuning hyperparameters (XGBoost)...")
        self._tune_xgboost()

        print("Hyperparameter tuning complete.")
        return pd.DataFrame(self.cv_results_summary)

In [3]:
class ThresholdTuner:

    def __init__(
        self,
        best_estimators,
        X_val_lr,
        y_val_lr,
        X_val_dt,
        y_val_dt,
    ):
        self.best_estimators = best_estimators

        # Validation data splits
        self.X_val_lr = X_val_lr
        self.y_val_lr = y_val_lr
        self.X_val_dt = X_val_dt
        self.y_val_dt = y_val_dt

        # Storage for optimal thresholds and results
        self.optimal_thresholds = {}
        self.tuning_results = []

    def _get_validation_data(self, model_name):
        if model_name == "Logistic Regression":
            return self.X_val_lr, self.y_val_lr
        else:
            return self.X_val_dt, self.y_val_dt

    def tune_thresholds(self, target_metric="f1"):
        # Threshold iterrated between 0.1 to 0.5 to find most optimal value.
        thresholds = np.linspace(0.10, 0.50, 41)

        for name, model in self.best_estimators.items():
            X_val, y_val = self._get_validation_data(name)
            probs = model.predict_proba(X_val)[:, 1]

            best_thresh = 0.5
            best_score = 0.0

            # Sweep candidate thresholds
            for thresh in thresholds:
                preds = (probs >= thresh).astype(int)
                score = (
                    f1_score(y_val, preds, zero_division=0)
                    if target_metric == "f1"
                    else recall_score(y_val, preds, zero_division=0)
                )

                if score > best_score:
                    best_score = score
                    best_thresh = thresh

            self.optimal_thresholds[name] = best_thresh

            # Record final validation metrics at the selected optimal threshold
            optimal_preds = (probs >= best_thresh).astype(int)
            self.tuning_results.append(
                {
                    "Model": name,
                    "Optimal Threshold": round(best_thresh, 2),
                    "Accuracy": round(accuracy_score(y_val, optimal_preds), 4),
                    "Precision": round(
                        precision_score(
                            y_val, optimal_preds, zero_division=0
                        ),
                        4,
                    ),
                    "Recall": round(
                        recall_score(y_val, optimal_preds, zero_division=0), 4
                    ),
                    "F1-Score": round(
                        f1_score(y_val, optimal_preds, zero_division=0), 4
                    ),
                    "ROC-AUC": round(roc_auc_score(y_val, probs), 4),
                }
            )

        return pd.DataFrame(self.tuning_results)

    def plot_confusion_matrices(self):
        # Confusion matrix foe each model
        if not self.optimal_thresholds:
            raise ValueError(
                "Must run tune_thresholds() before plotting confusion matrices."
            )

        n_models = len(self.best_estimators)
        fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))

        if n_models == 1:
            axes = [axes]

        for ax, (name, model) in zip(axes, self.best_estimators.items()):
            X_val, y_val = self._get_validation_data(name)
            thresh = self.optimal_thresholds[name]

            probs = model.predict_proba(X_val)[:, 1]
            preds = (probs >= thresh).astype(int)

            cm = confusion_matrix(y_val, preds)

            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                cbar=False,
                ax=ax,
                xticklabels=["No Stroke", "Stroke"],
                yticklabels=["No Stroke", "Stroke"],
            )
            ax.set_title(f"{name}\nThreshold: {thresh:.2f}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("Actual")

        plt.tight_layout()
        plt.show()